In [0]:
%run ./00_config

In [0]:

from pyspark.sql.functions import current_timestamp
from pyspark.sql import Row

# =========================
# TRACKING
# =========================

def get_last_write(table_name):
    df = spark.sql(f"""
        SELECT last_write_date
        FROM {TRACKING_TABLE}
        WHERE table_name = '{table_name}'
        ORDER BY last_write_date DESC
        LIMIT 1
    """)

    row = df.collect()

    if not row or row[0]["last_write_date"] is None:
        return "1900-01-01 00:00:00"

    # Always return as string, never datetime object
    return row[0]["last_write_date"].strftime("%Y-%m-%d %H:%M:%S")

    
def update_tracking(table_name, records):
    if not records:
        return

    max_date = max(
        r.get("write_date")
        for r in records
        if r.get("write_date")
    )

    spark.sql(f"""
        MERGE INTO {TRACKING_TABLE} t
        USING (SELECT '{table_name}' AS table_name, CAST('{max_date}' AS TIMESTAMP) AS last_write_date) u
        ON t.table_name = u.table_name
        WHEN MATCHED THEN UPDATE SET t.last_write_date = u.last_write_date
        WHEN NOT MATCHED THEN INSERT (table_name, last_write_date)
        VALUES (u.table_name, u.last_write_date)
    """)